# 02 · Procesamiento y Feature Engineering

**Prueba Técnica Data Scientist · Ferreycorp** — Modelo de Propensión de Compra

Fase 2 de [`docs/brief-metodologia.md`](../docs/brief-metodologia.md). El notebook 01 cerró el
EDA con **doce de los trece controles de tratamiento en OK y uno pendiente**: el **E10**
—*escalado y codificación ajustados solo en train*— no se podía verificar porque todavía no
existía el `Pipeline`. Este notebook lo construye y lo cierra.

El objetivo no es «crear variables». Es entregar una matriz modelable donde el **límite del
ajuste** esté demostrado: todo estadístico aprendido sale de train y de ningún otro sitio.

| Capa | Módulo |
|---|---|
| Carga, splits, validación del contrato | `ferreyros.data` |
| Las cinco familias de variables derivadas | `ferreyros.features` |
| Codificación, escalado y `Pipeline` | `ferreyros.preprocess` |
| Celda de control y auditoría de los 13 controles | `ferreyros.audit` |

La lógica vive en `src/ferreyros/`; aquí solo se orquesta y se interpreta. Cada bloque cierra
con su **consecuencia** declarada.

In [1]:
import sys
from pathlib import Path

# Permite ejecutar el notebook sin instalar el paquete (uv run jupyter / VS Code).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from ferreyros import audit, config as cfg, data, features
from ferreyros import preprocess as prep

pd.set_option("display.max_columns", 60, "display.width", 180)
pd.set_option("display.max_colwidth", 90)

print("raíz del proyecto:", ROOT)

raíz del proyecto: /Users/jessieblake/Developer/Projects/Ferreyros


---
## 0 · Celda de control

Misma celda que abre el notebook 01, por la misma razón: si una cifra de referencia no cuadra,
el error es de carga y contaminaría todo lo que viene después. Se ejecuta sobre el frame
**crudo**, antes de añadir cualquier columna derivada.

In [2]:
df_raw = data.load_raw()
control = audit.control_cell(df_raw)

print(f"Cifras verificadas: {len(control)}  |  discrepancias: {control.attrs['n_fallas']}")
assert control.attrs["all_ok"], "La celda de control no cuadra: revisar la carga antes de seguir"
control.head(8)

Cifras verificadas: 44  |  discrepancias: 0


,cifra,observado,esperado,veredicto
0,n_rows,58693.0,58693.0,OK
1,n_cols,24.0,24.0,OK
2,n_clients,500.0,500.0,OK
3,n_days,730.0,730.0,OK
4,day_min,1.0,1.0,OK
5,day_max,730.0,730.0,OK
6,n_nulls,0.0,0.0,OK
7,n_duplicates,0.0,0.0,OK


**Consecuencia:** las 44 cifras cuadran. Partimos del mismo dataset sobre el que se escribieron
los briefs, así que la Fase 2 es comparable con la Fase 1.

In [3]:
# `precio_pagado` y `gasto` (control A7). Son resultados de la visita: sirven para construir
# el ticket histórico, pero nunca entran como features de M1.
df = data.add_paid_price_and_spend(df_raw)
print("columnas:", df_raw.shape[1], "→", df.shape[1])

columnas: 24 → 26


---
## 1 · Las dos nociones de «pasado», y por qué confundirlas es la fuga clásica

En esta fase hay dos cosas que miran hacia atrás y **no son lo mismo**:

| | Qué es | Dónde debe calcularse | Control |
|---|---|---|---|
| **Historial del cliente** | Sus visitas anteriores | Sobre **toda** la línea de tiempo, hacia atrás | E6 |
| **Estadísticos de referencia** | Medias, máximos, bordes de quintil | Sobre **train únicamente** | E10 |

El historial de un cliente en el período de test **sí** puede usar sus visitas de train: eso
es información pasada, no fuga. Lo que no puede cruzar el corte es un parámetro *ajustado*: si
la media contra la que comparo el precio de una marca se calcula sobre train+test, la
distribución del futuro está moldeando una variable del pasado.

Tres grupos de features necesitaban un valor de referencia que no está en la fila:
`desviacion_precio_j` (la media histórica de la marca), `descuento_implicito_j` (el precio de
lista, o máximo histórico) y los quintiles demográficos `edad_bin` / `ingreso_bin`. Los tres
viven ahora en un objeto ajustado, no en una expresión dentro de la función.

In [4]:
# El orden importa: se parte ANTES de ajustar nada.
train_raw, test_raw = data.temporal_split(df)
print(f"train: días {train_raw[cfg.KEY_TIME].min()}–{train_raw[cfg.KEY_TIME].max()}  "
      f"({len(train_raw):,} filas)")
print(f"test:  días {test_raw[cfg.KEY_TIME].min()}–{test_raw[cfg.KEY_TIME].max()}  "
      f"({len(test_raw):,} filas)")

reference = features.fit_reference(train_raw)
reference.describe()

train: días 1–550  (45,497 filas)
test:  días 551–730  (13,196 filas)


,estadistico,valor
0,precio_medio[precio_marca_1],1.3895
1,precio_medio[precio_marca_2],1.7797
2,precio_medio[precio_marca_3],1.9925
3,precio_medio[precio_marca_4],2.1366
4,precio_medio[precio_marca_5],2.6331
5,precio_max[precio_marca_1],1.59
6,precio_max[precio_marca_2],1.9
7,precio_max[precio_marca_3],2.07
8,precio_max[precio_marca_4],2.26
9,precio_max[precio_marca_5],2.71


In [5]:
# ¿El límite del ajuste es material o cosmético? Comparamos cada estadístico contra el valor
# que tendría ajustado sobre train+test.
ref_check = features.verify_reference_fit(
    features.build_features(df, reference=reference)
)
display(ref_check)
print(ref_check.attrs["lectura"])
assert ref_check.attrs["e10_features_ok"], "Los estadísticos de referencia no son solo de train"

,estadistico,ajustado,solo_train,train+test,coincide_con_train,estado
0,precio_medio[precio_marca_1],1.389478,1.389478,1.392074,True,OK
1,precio_medio[precio_marca_2],1.779651,1.779651,1.780999,True,OK
2,precio_medio[precio_marca_3],1.992452,1.992452,2.006789,True,OK
3,precio_medio[precio_marca_4],2.136648,2.136648,2.159945,True,OK
4,precio_medio[precio_marca_5],2.633089,2.633089,2.654798,True,OK
5,precio_max[precio_marca_1],1.59,1.59,1.59,True,OK
6,precio_max[precio_marca_2],1.9,1.9,1.9,True,OK
7,precio_max[precio_marca_3],2.07,2.07,2.14,True,OK
8,precio_max[precio_marca_4],2.26,2.26,2.26,True,OK
9,precio_max[precio_marca_5],2.71,2.71,2.8,True,OK


12 estadisticos ajustados solo en train (dias 1-550, 45,497 filas); 8 difieren del valor que tendrian ajustados sobre train+test, asi que la distincion no es cosmetica.


**Consecuencia, y es un hallazgo concreto, no una formalidad:** 8 de los 12 estadísticos
cambian según se ajusten sobre train o sobre todo el frame. El caso claro son las marcas 3 y 5:
su precio máximo es **2.07 y 2.71 en train**, pero **2.14 y 2.80 si se mira todo el dataset**.
Esos precios altos solo ocurren en el período de test.

Con la implementación anterior, el `descuento_implicito` de *cada fila de train* se medía contra
un precio de lista que todavía no había existido. El modelo habría visto el futuro por una vía
que ninguna matriz de features delata: el valor de la columna. Ajustar sobre train lo cierra.

---
## 2 · Las cinco familias de variables derivadas

§3 del [brief de problema](../docs/brief-problema.md): precio competitivo, promoción, historial
de cliente, temporal y demográfica. Se construyen sobre el frame **completo** —el historial
necesita la línea de tiempo entera— pero con la referencia ajustada en train.

In [6]:
feat = features.build_features(df, reference=reference)
print("columnas:", df.shape[1], "→", feat.shape[1],
      f"({len(features.all_feature_names())} variables derivadas)")
features.feature_summary(feat)

columnas: 26 → 68 (42 variables derivadas)


,familia,feature,pct_nulos,media,min,max
0,precio,precio_min_dia,0.0,1.389,1.100,1.590
1,precio,precio_medio_dia,0.0,1.999,1.872,2.102
2,precio,precio_max_dia,0.0,2.655,2.160,2.800
3,precio,rango_precios_dia,0.0,1.266,0.800,1.590
4,precio,precio_relativo_1,0.0,0.696,0.561,0.781
5,precio,precio_relativo_2,0.0,0.890,0.669,0.993
6,precio,precio_relativo_3,0.0,1.004,0.938,1.076
7,precio,precio_relativo_4,0.0,1.081,0.924,1.176
8,precio,precio_relativo_5,0.0,1.328,1.114,1.427
9,precio,desviacion_precio_1,0.0,0.002,-0.208,0.144


Los nulos de la familia `historial` son **esperados y correctos**: en las primeras visitas de un
cliente no existe historial. El 8.5% de `lealtad_marca_j` y `dias_desde_ultima_compra` son las
visitas previas a la primera compra. No se imputan con la media (control E7); el §5 muestra qué
se hace con ellos.

### 2.1 · Verificación de fuga en el historial (E6)

Tres pruebas sobre la matriz construida, en lugar de una declaración de intención.

In [7]:
leak = features.verify_no_leakage(feat)
display(leak)
assert leak.attrs["sin_fuga"], "Fuga detectada en las features de historial"
print("Sin fuga: ninguna feature de historial contiene información de la visita actual.")

,verificacion,features_evaluadas,violaciones,detalle,estado
0,Primera visita sin historial (features = 0 o NaN),13,0,ninguna,OK
1,n_compras_previas excluye la compra de la fila actual,1,0,cumsum() - fila actual,OK
2,lealtad_marca no reacciona a la compra actual,5,0,53 primeras visitas con compra evaluadas,OK


Sin fuga: ninguna feature de historial contiene información de la visita actual.


**Consecuencia:** las features de historial son utilizables en M1. La prueba que más importa es
la tercera: entre los clientes que compran en su **primera** visita, la lealtad previa a la
marca elegida es nula. Si reaccionara, la variable estaría leyendo el resultado de la fila.

### 2.2 · Matriz admisible para M1 (E1, E2, E3)

In [8]:
X_train = features.feature_matrix(feat[feat[cfg.KEY_TIME] <= cfg.SPLIT_DAY])
X_test = features.feature_matrix(feat[feat[cfg.KEY_TIME] > cfg.SPLIT_DAY])[list(X_train.columns)]

print("matriz M1:", X_train.shape[1], "features admisibles")
for forbidden in [cfg.TARGET_BRAND, cfg.TARGET_QUANTITY, *cfg.EXCLUDED_COLS,
                  "precio_pagado", "gasto"]:
    assert forbidden not in X_train.columns, f"{forbidden} no puede ser feature de M1"
    print(f"  excluida: {forbidden}")

matriz M1: 60 features admisibles
  excluida: id_marca
  excluida: cantidad
  excluida: tamanio_ciudad
  excluida: precio_pagado
  excluida: gasto


**Consecuencia:** `id_marca` y `cantidad` son **resultados** de la visita, no condiciones
previas; `tamanio_ciudad` está excluida por indicación del diccionario; `precio_pagado` y
`gasto` se derivan de la marca elegida, así que son resultados también. Si entraran, las
métricas saldrían casi perfectas y el modelo sería inútil en producción.

---
## 3 · Roles de columna: el tratamiento se declara, no se adivina por `dtype`

Todas las columnas son numéricas, así que un `dtype` no distingue una categoría ordinal de una
cuenta. El tratamiento se declara por rol.

In [9]:
prep.column_roles(X_train)

,rol,n_columnas,tratamiento,por_que,columnas
0,nominal,3,OneHotEncoder(handle_unknown='ignore'),categoria sin orden,"ocupacion, ultima_marca_comprada, dia_semana"
1,ordinal,3,mediana + StandardScaler,el orden es informacion; se conserva como numero,"nivel_educacion, edad_bin, ingreso_bin"
2,binaria,9,passthrough,ya es 0/1; escalarla solo complica la lectura,"genero, estado_civil, ultima_cantidad_comprada, promo_en_ultima_marca ..."
3,historial_con_vacios,10,constante 0.0 + indicador + StandardScaler,"el vacio es ausencia de historial (E7), no un valor desconocido","tasa_compra_historica, dias_desde_ultima_compra, cantidad_media_historica, ticket_medi..."
4,numerica,35,mediana + StandardScaler,continua y completa,"precio_min_dia, precio_medio_dia, precio_max_dia, rango_precios_dia ..."


El rol **`historial_con_vacios`** es la decisión de diseño de esta fase. El brief de
alineamiento (E7) dice que la ausencia de historial «no se imputa». Imputar la media diría
«este cliente compra como el promedio», que es falso: el cliente **todavía no ha comprado**.

El tratamiento es constante + `add_indicator=True`: el relleno es un marcador identificable y
el **indicador** es la variable que el modelo lee. Así la ausencia llega como información
explícita en lugar de disfrazarse de observación.

---
## 4 · El `Pipeline`, ajustado solo sobre train

`prep.prepare()` reproduce en una llamada el orden de este notebook —partir, ajustar la
referencia en train, construir features sobre toda la línea de tiempo, ajustar el preprocesador
en train— y devuelve las matrices listas para la Fase 4.

In [10]:
prepared = prep.prepare(df)
prepared.shapes

,conjunto,filas,cols_entrada,cols_salida,tasa_compra,clientes
0,train,45497,60,83,0.2498,500
1,test,13196,60,83,0.2480,500


In [11]:
# La narrativa de arriba y la función de una línea deben coincidir exactamente.
assert list(prepared.X_train.columns) == list(X_train.columns)
assert prepared.X_train.equals(X_train) and prepared.X_test.equals(X_test)
print("La matriz construida paso a paso y la de prepare() son idénticas.")

prep.preprocessing_summary(prepared).to_frame("valor")

La matriz construida paso a paso y la de prepare() son idénticas.


,valor
filas totales,58693
corte temporal (dia),550
filas train / test,"45,497 / 13,196"
features de entrada,60
features tras codificar,83
columnas indicadoras de ausencia,10
columnas one-hot,16
nulos tras preprocesar,0


**Consecuencia:** 60 features de entrada → **83 columnas** modelables: 10 indicadores de
ausencia de historial y 16 niveles one-hot (ocupación, última marca comprada —el nivel `0` es
«sin historial», una categoría propia— y día de la semana). Cero nulos a la salida, sin haber
imputado una sola media sobre una ausencia que significaba algo.

---
## 5 · Control E10, probado

El control que el notebook 01 dejó pendiente. Cuatro aserciones, y la segunda es la que le da
dientes: la primera también pasaría en un pipeline ajustado sobre todo, así que sin una prueba
de materialidad el control sería decorativo.

In [12]:
e10 = prep.verify_train_only_fit(prepared)
display(e10)
print(e10.attrs["lectura"])
assert e10.attrs["e10_ok"], "El preprocesamiento aprendió algo de test"

,control,verificacion,medida,cumple,estado
0,E10.1,La media del escalador es la media de train,desvio maximo 0.00e+00 en 48 columnas,True,OK
1,E10.2,Ajustar sobre train+test daria parametros distintos,48 de 48 medias cambiarian,True,OK
2,E10.3,transform(test) no modifica los parametros ajustados,desvio maximo 0.00e+00,True,OK
3,E10.4,Las categorias del codificador provienen de train,"16 categorias, todas de train; 0 niveles de test ignorados",True,OK


Pipeline ajustado sobre 45,497 filas de train (dias 1-550); 48 parametros cambiarian si se ajustara sobre las 58,693 filas, de modo que el limite del ajuste es material y no declarativo.


**Consecuencia:** el preprocesador aprendió de 45,497 filas de train y de ninguna otra. Las 48
medias del escalador cambiarían si se ajustara sobre las 58,693 filas, `transform(test)` no
mueve ningún parámetro, y las 16 categorías del codificador salen de train con
`handle_unknown='ignore'` para un nivel no visto. **E10 queda en OK con evidencia ejecutada.**

---
## 6 · Qué extrapola: la razón honesta de que la validación sea temporal

Un corte temporal garantiza que algunas features tengan en test valores que train nunca vio.
No es un defecto, pero sí determina qué modelos son seguros.

In [13]:
oor = prep.out_of_range_report(prepared)
display(oor)
print("features que extrapolan:", oor.attrs["n_features_extrapolan"], "de", X_train.shape[1])

,feature,rango_train,rango_test,pct_test_fuera_de_rango
0,semana,"[1, 79]","[79, 105]",98.0
1,mes,"[1, 19]","[19, 25]",88.4
2,precio_max_dia,"[2.16, 2.71]","[2.38, 2.8]",79.6
3,desviacion_precio_5,"[-0.199, 0.0292]","[-0.0961, 0.0634]",79.6
4,descuento_implicito_5,"[0, 0.221]","[-0.0332, 0.122]",79.6
5,precio_marca_5,"[2.11, 2.71]","[2.38, 2.8]",79.6
6,precio_medio_dia,"[1.87, 2.06]","[1.96, 2.1]",35.9
7,desviacion_precio_3,"[-0.0615, 0.0389]","[-0.0113, 0.0741]",15.1
8,descuento_implicito_3,"[0, 0.0966]","[-0.0338, 0.0483]",15.1
9,precio_marca_3,"[1.87, 2.07]","[1.97, 2.14]",15.1


features que extrapolan: 16 de 60


**Consecuencia — dos lecturas, y ambas condicionan la Fase 4:**

1. **`semana` y `mes` son índices de tendencia.** El 98% y el 88% de las filas de test caen
   fuera del rango de train, porque el tiempo avanza. Una regresión lineal les extrapolaría una
   pendiente sobre un índice que nunca observó. Se conservan como diagnóstico, pero **la
   estacionalidad entra por `dia_semana`** (one-hot, acotado) y la decisión sobre `semana`/`mes`
   se toma con la métrica en la mano, no por costumbre.
2. **El precio de la marca 5 sube en el período de test** (hasta 2.80 frente a 2.71 en train) y
   afecta al 79.6% de las filas de test. Es exactamente la situación que el problema de pricing
   plantea: decidir sobre precios que no se han observado. Confirma que la escalera de precios
   discreta de B1 debe construirse sobre los puntos observados y que extrapolar fuera de ella
   sería inventar demanda.

---
## 7 · Auditoría: los trece controles de tratamiento

§3 del [brief de alineamiento](../docs/brief-alineamiento-eda.md), ahora con el `Pipeline`
disponible.

In [14]:
aud = audit.treatment_audit(
    df_raw,
    feature_cols=list(prepared.X_train.columns),
    featured_df=feat,
    prepared=prepared,            # cierra E10
    notebook_runs_clean=True,     # este notebook corre de extremo a extremo
)
display(aud)
print("bloqueantes fallados:", aud.attrs["bloqueantes_fallados"],
      "| pendientes:", aud.attrs["pendientes"])
assert aud.attrs["bloqueantes_fallados"] == 0, "Hay controles bloqueantes en FALLA"
assert aud.attrs["pendientes"] == 0, "Quedan controles pendientes"

,control,verificacion,severidad,estado,evidencia
0,E1,tamanio_ciudad excluida,Bloqueante,OK,Verificado contra la matriz de features
1,E2,id_marca fuera de las features de M1,Bloqueante,OK,"Es resultado de la visita, no condicion previa"
2,E3,cantidad fuera de las features de M1,Bloqueante,OK,"Es resultado de la visita, no condicion previa"
3,E4,ultima_marca_comprada validada como lag,Bloqueante,OK,Coincidencia con shift(1) por cliente: 1.0
4,E5,ultima_cantidad_comprada reinterpretada,Alta,OK,Binaria (0/1): el diccionario la nombra mal; es el lag del indicador de compra
5,E6,Features de historial retrospectivas,Bloqueante,OK,"3 pruebas sobre la matriz construida: primera visita sin historial, n_compras_previas ..."
6,E7,ultima_marca_comprada=0 como categoria,Alta,OK,"44,133 filas sin historial; no se imputan"
7,E8,"Split temporal, no aleatorio",Bloqueante,OK,Corte en dia_visita <= 550
8,E9,Split sin sesgo de composicion,Alta,OK,"train 45,497 @ 0.2498 / test 13,196 @ 0.248"
9,E10,Escalado y codificacion solo en train,Bloqueante,OK,"Pipeline ajustado sobre 45,497 filas de train (dias 1-550): 4 pruebas en preprocess.ve..."


bloqueantes fallados: 0 | pendientes: 0


**Consecuencia:** **13 de 13 controles en OK, cero pendientes.** El notebook 01 cerró con E10
pendiente por diseño; la Fase 2 lo cierra con cuatro pruebas ejecutadas. Ningún control de
tratamiento queda apoyado en una declaración.

---
## 8 · Síntesis: qué entrega la Fase 2

| Entrega | Estado | Evidencia |
|---|---|---|
| Matriz modelable | 83 columnas desde 60 features, 0 nulos | `prepare().shapes` |
| Features sin fuga de resultado | 3 pruebas, 0 violaciones | `features.verify_no_leakage` |
| Estadísticos ajustados solo en train | 12 estadísticos, 8 materialmente distintos | `features.verify_reference_fit` |
| Preprocesamiento ajustado solo en train | 4 pruebas, control E10 en OK | `preprocess.verify_train_only_fit` |
| Ausencia de historial como información | 10 indicadores, ninguna media imputada | `preprocess.column_roles` |
| Split temporal sin sesgo | 500 clientes y misma tasa a ambos lados | `data.split_composition` |

**Lo que esta fase descubrió y no estaba previsto:** tres estadísticos de feature se estaban
ajustando sobre train+test. Era una fuga invisible —ninguna columna prohibida aparecía en la
matriz— y los precios máximos de las marcas 3 y 5 muestran que era real, no teórica.

**Lo que queda abierto para la Fase 4, declarado y no escondido:** 16 features extrapolan fuera
del rango de train. Las de tendencia (`semana`, `mes`) son una decisión de modelado pendiente;
las de precio son el problema de negocio mismo.

**Siguiente paso:** Fase 3 — segmentación K-Means sobre la demografía a nivel cliente
(`eda.segments.segmentation_matrix`), con el segmento incorporado como feature de M1 y, sobre
todo, como unidad de acción comercial.